# Dataset Build Colab Notebook

## 0. Runtime Setup
### 0.1 Configure Dataset Build Inputs

    What this step does: defines the repository revision, requested dataset splits, and mirrored runtime/Drive project roots used by every later cell.

    Required inputs: operator-edited REPO_URL, REPO_REF, and DATASET_SPLITS values. WORKTREE_ROOT and DRIVE_PROJECT_ROOT are fixed Colab project roots and should only change if the project-wide Colab layout changes.

    Produced outputs: configured constants printed for review before any Drive or filesystem operation runs.

    When this step may be skipped: only when these constants are already defined in the current runtime exactly as intended for this build.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "master"
DATASET_SPLITS = ("train", "val", "test")  # Single split example: ("val",)

WORKTREE_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")
print("Worktree root:", WORKTREE_ROOT)
print("Drive project root:", DRIVE_PROJECT_ROOT)
print("Dataset splits:", DATASET_SPLITS)


### 0.2 Mount Google Drive

What this step does: mounts Google Drive at /content/drive and confirms the mirrored project root can be addressed under MyDrive.

Required inputs: DRIVE_PROJECT_ROOT from step 0.1 and an authenticated Colab Drive session.

Produced outputs: a mounted Drive filesystem with DRIVE_PROJECT_ROOT.parent present.

When this step may be skipped: only when Drive is already mounted and DRIVE_PROJECT_ROOT.parent has been verified in this runtime.


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive MyDrive root is missing: {DRIVE_PROJECT_ROOT.parent}")
print("Drive mounted:", DRIVE_PROJECT_ROOT.parent)


### 0.3 Ensure zstd Is Available

What this step does: checks for the zstd command and installs the Debian package when the Colab runtime does not provide it.

Required inputs: Colab apt package access and sudo permission in the runtime.

Produced outputs: a runtime where zstd --version succeeds before any tar --zstd command is used.

When this step may be skipped: only when shutil.which("zstd") already finds zstd and zstd --version has succeeded in this runtime.


In [ ]:
import shutil

if shutil.which("zstd") is None:
    !sudo apt-get update

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to update apt package index.")
    
    !sudo apt-get install -y zstd
    
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to install zstd.")
    print("Installed zstd.")
else:
    print("zstd is already available.")
     

!zstd --version

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("zstd command is not available after preflight.")

### 0.4 Acquire Repository

What this step does: removes any old runtime checkout, clones REPO_URL into WORKTREE_ROOT, checks out REPO_REF, and prints the resolved commit.

Required inputs: REPO_URL, REPO_REF, writable /content, and network access to the repository.

Produced outputs: a fresh repository checkout at WORKTREE_ROOT with the requested revision checked out.

When this step may be skipped: only when WORKTREE_ROOT already contains the same requested revision and the operator intentionally wants to reuse it.


In [ ]:
%cd /content/
!rm -rf {WORKTREE_ROOT}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to remove existing directory: {WORKTREE_ROOT}")

!git clone {REPO_URL} {WORKTREE_ROOT}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to clone repository.")

!git -C {WORKTREE_ROOT} checkout {REPO_REF}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to checkout revision {REPO_REF}.")

print(f"Repository cloned at {WORKTREE_ROOT}")

!git -C {WORKTREE_ROOT} rev-parse HEAD

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to determine checked out revision.")

print("Repository ready:", WORKTREE_ROOT)

### 0.5 Install Python Dependencies

What this step does: runs pip from the repository checkout and installs the Colab requirements file.

Required inputs: WORKTREE_ROOT containing requirements-colab.txt and Python package index access.

Produced outputs: the notebook runtime has the project Colab Python dependencies installed.

When this step may be skipped: only when these exact requirements have already been installed in the active Colab runtime.


In [ ]:
%cd {WORKTREE_ROOT}

%pip install --upgrade pip

%pip install -r "requirements-colab.txt"

print("Repository requirements installed. Please, restart the session.")

### 0.6 Add Source Tree To Python Path

What this step does: changes to the repository checkout and prepends the local src tree to sys.path for notebook imports.

Required inputs: WORKTREE_ROOT with a src directory from the checked-out repository.

Produced outputs: WORKTREE_ROOT / "src" is available on sys.path for subsequent imports.

When this step may be skipped: only when the same src path is already present on sys.path in this runtime.


In [ ]:
import sys

%cd {WORKTREE_ROOT}

if str(WORKTREE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(WORKTREE_ROOT / "src"))

print("Repository src directory added to sys.path:", WORKTREE_ROOT / "src")

### 0.7 Import Dataset Workflow API And Build Runtime Plan

What this step does: imports the Dataset workflow API and builds the runtime plan that describes Drive input paths, runtime target paths, visible shell commands, and verifier contracts.

Required inputs: the installed dependencies, sys.path entry from steps 0.5 and 0.6, WORKTREE_ROOT, DRIVE_PROJECT_ROOT, and DATASET_SPLITS.

Produced outputs: dataset_runtime_plan plus workflow functions used by later cells; no core or data modules are imported by the notebook.

When this step may be skipped: only when dataset_runtime_plan already reflects the current WORKTREE_ROOT, DRIVE_PROJECT_ROOT, and DATASET_SPLITS.


In [ ]:
from text_to_sign_production.workflows.dataset import (
    build_dataset_publish_plan,
    build_dataset_quality_readiness_summary,
    build_dataset_runtime_plan,
    run_dataset_workflow,
    summarize_dataset_publish_plan,
    summarize_dataset_workflow_outputs,
    validate_dataset_inputs,
    verify_dataset_runtime_inputs,
    write_dataset_archive_member_list,
)

dataset_runtime_plan = build_dataset_runtime_plan(
    project_root=WORKTREE_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
    splits=DATASET_SPLITS,
)
print("Dataset workflow API ready:", build_dataset_runtime_plan.__module__)
print("Runtime project root:", dataset_runtime_plan.project_root)
print("Drive project root:", dataset_runtime_plan.drive_project_root)
print("Dataset splits:", dataset_runtime_plan.splits)


## 1. Restore Raw How2Sign Inputs
### 1.1 Locate Drive Translation CSVs

    What this step does: displays the workflow-provided raw translation transfer specs for each requested split.

    Required inputs: dataset_runtime_plan after Drive has been mounted.

    Produced outputs: RAW_TRANSLATION_SPECS mapping each split to verified Drive source and runtime target paths.

    When this step may be skipped: only when RAW_TRANSLATION_SPECS already comes from the current dataset_runtime_plan.


In [ ]:
RAW_TRANSLATION_SPECS = dataset_runtime_plan.raw_translation_specs
for split, spec in RAW_TRANSLATION_SPECS.items():
    print(f"{split}: {spec.source_path} -> {spec.target_path} ({spec.expected_input_bytes} bytes)")


### 1.2 Copy Translation CSVs To Runtime

What this step does: executes each workflow-provided visible translation copy command with byte progress.

Required inputs: RAW_TRANSLATION_SPECS from dataset_runtime_plan.

Produced outputs: RAW_TRANSLATION_TARGETS mapping each split to a workflow-verified runtime CSV.

When this step may be skipped: only when the runtime translation CSVs already match the Drive inputs for the requested split scope.


In [ ]:
RAW_TRANSLATION_TARGETS = {}
for split, transfer_spec in RAW_TRANSLATION_SPECS.items():
    spec = transfer_spec.copy_command
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)

    RAW_TRANSLATION_TARGETS[split] = transfer_spec.target_path

print("Raw translations copied:", RAW_TRANSLATION_TARGETS)


### 1.3 Locate Drive Raw BFH Archives

What this step does: displays the workflow-provided raw BFH archive restore specs for each requested split.

Required inputs: dataset_runtime_plan after Drive has been mounted.

Produced outputs: RAW_BFH_ARCHIVE_SPECS mapping each split to a verified source-format Drive .tar.zst archive spec.

When this step may be skipped: only when RAW_BFH_ARCHIVE_SPECS already comes from the current dataset_runtime_plan.


In [ ]:
RAW_BFH_ARCHIVE_SPECS = dataset_runtime_plan.raw_bfh_archive_specs
for split, spec in RAW_BFH_ARCHIVE_SPECS.items():
    print(f"{split}: {spec.archive_path} -> {spec.extraction_root} ({spec.expected_input_bytes} bytes)")


### 1.4 Extract Raw BFH Archives

What this step does: executes each workflow-provided visible raw BFH extraction command with byte progress.

Required inputs: RAW_BFH_ARCHIVE_SPECS, zstd support, and writable runtime raw BFH directories.

Produced outputs: RAW_BFH_SPLIT_ROOTS mapping each split to the workflow-expected raw BFH split root.

When this step may be skipped: only when the runtime split roots already contain the extracted source archive contents for the requested splits. Raw BFH archives are source-format How2Sign/OpenPose archives. The workflow extracts each archive into the canonical runtime BFH keypoints root so the split directory contains `openpose_output/json` and `openpose_output/video`.


In [ ]:
%cd {WORKTREE_ROOT}
RAW_BFH_SPLIT_ROOTS = {}
for split, archive_spec in RAW_BFH_ARCHIVE_SPECS.items():
    spec = archive_spec.extract_command
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)

    RAW_BFH_SPLIT_ROOTS[split] = archive_spec.expected_split_root

print("Raw BFH archives extracted:", RAW_BFH_SPLIT_ROOTS)


### 1.5 Verify Extracted Raw BFH Layout

What this step does: asks the workflow verifier to check that each extracted raw BFH split contains the expected openpose_output/json and openpose_output/video layout.

Required inputs: dataset_runtime_plan and extracted raw archive contents.

Produced outputs: RAW_KEYPOINT_LAYOUTS mapping each split to its verified runtime raw BFH split root.

When this step may be skipped: only when RAW_KEYPOINT_LAYOUTS has already been verified for the same requested split scope in this runtime.


In [ ]:
verify_dataset_runtime_inputs(dataset_runtime_plan)
RAW_KEYPOINT_LAYOUTS = {
    split: spec.expected_split_root
    for split, spec in dataset_runtime_plan.raw_bfh_archive_specs.items()
}
print("Raw BFH layouts restored:", RAW_KEYPOINT_LAYOUTS)


## 2. Run Dataset Workflow
### 2.1 Configure Dataset Workflow

    What this step does: selects the DatasetWorkflowConfig produced by the workflow runtime plan.

    Required inputs: dataset_runtime_plan and restored raw inputs.

    Produced outputs: dataset_config ready for explicit validation and execution.

    When this step may be skipped: only when an equivalent dataset_config has already been selected for this runtime and split scope.


In [ ]:
dataset_config = dataset_runtime_plan.workflow_config
print("Dataset workflow configured for:", dataset_config.splits)


### 2.2 Validate Dataset Workflow Inputs

What this step does: runs the workflow input validator before generating derived dataset outputs.

Required inputs: dataset_config and the restored runtime translation, keypoint JSON, and source video inputs.

Produced outputs: a completed validation check for the configured dataset workflow inputs.

When this step may be skipped: only when the same dataset_config has already passed validation after the current input restore.


In [ ]:
validate_dataset_inputs(dataset_config)
print("Dataset workflow inputs validated:", DATASET_SPLITS)

### 2.3 Run Dataset Workflow

What this step does: runs raw manifest, normalization, filtering, final manifest export, and processed sample export stages.

Required inputs: validated dataset_config and restored raw runtime inputs.

Produced outputs: dataset_result containing generated manifests, reports, sample directories, and archive member metadata.

When this step may be skipped: only when dataset_result already reflects the same restored inputs, configuration, and code revision.


In [ ]:
dataset_result = run_dataset_workflow(dataset_config)
print("Dataset workflow complete:", dataset_result.data_root)

### 2.4 Inspect Dataset Build Summary

What this step does: verifies workflow outputs and prints a compact workflow-owned operational digest.

Required inputs: dataset_result from the completed dataset workflow.

Produced outputs: dataset_output_summary with verified processed manifests, processed sample counts, validation issue counts, raw source issue counts, dropped sample counts, top drop reasons, and report paths.

When this step may be skipped: only when these result postconditions have already been checked for this exact workflow run.


In [ ]:
dataset_output_summary = summarize_dataset_workflow_outputs(dataset_result)

print("Dataset Build Summary")
print("---------------------")
print("splits:", ", ".join(dataset_output_summary.splits))
print("validation errors:", dataset_output_summary.validation_error_count)
print("validation warnings:", dataset_output_summary.validation_warning_count)

print()
print("Processed samples")
for split, member_count in dataset_output_summary.processed_sample_archive_member_counts.items():
    print(f"- {split}: {member_count} manifest-referenced samples")

print()
print("Processed manifests")
for split, processed_manifest_path in dataset_output_summary.processed_manifest_paths.items():
    print(f"- {split}: {processed_manifest_path}")

print()
print("Validation issue counts")
if dataset_output_summary.validation_issue_counts:
    for stage, issue_count in dataset_output_summary.validation_issue_counts.items():
        print(f"- {stage}: {issue_count}")
else:
    print("- none")

print()
print("Raw source issue counts")
for split, issue_count in dataset_output_summary.raw_source_issue_counts.items():
    print(f"- {split}: {issue_count}")

print()
print("Dropped samples")
for split, dropped_count in dataset_output_summary.dropped_sample_counts.items():
    print(f"- {split}: {dropped_count}")

print()
print("Top drop reasons")
for split, reasons in dataset_output_summary.top_drop_reasons.items():
    if reasons:
        rendered_reasons = "; ".join(f"{reason} = {count}" for reason, count in reasons)
    else:
        rendered_reasons = "none"
    print(f"- {split}: {rendered_reasons}")

print()
print("Reports")
for label, report_path in dataset_output_summary.report_paths.items():
    print(f"- {label}: {report_path}")

# --- Dataset quality readiness summary ---
dataset_quality_readiness = build_dataset_quality_readiness_summary(dataset_output_summary)

print()
print("Dataset quality readiness")
print("-------------------------")
print("Quality config:", dataset_quality_readiness.quality_config_path)
print("Quality config sha256:", dataset_quality_readiness.quality_config_hash)

print()
print("Quality reports:")
for split, path in dataset_quality_readiness.quality_report_paths.items():
    print(f"- {split}: {path}")

print()
print("Leakage report:")
for key, path in dataset_quality_readiness.leakage_report_paths.items():
    print(f"- {key}: {path}")
print("- blocking_for_complete:", dataset_quality_readiness.blocking_for_complete)

print()
print("Tier manifests:")
for split_or_key, tier_paths in dataset_quality_readiness.tier_manifest_paths.items():
    for tier, path in tier_paths.items():
        print(f"- {tier}/{split_or_key}: {path}")

print()
print("Tier counts:")
for split, counts in dataset_quality_readiness.tier_counts.items():
    rendered = ", ".join(f"{tier}={n}" for tier, n in counts.items())
    print(f"- {split}: {rendered}")

print()
print("Complete-mode Dataset gate:", dataset_quality_readiness.complete_dataset_gate)


## 3. Publish Dataset Outputs
### 3.1 Build Dataset Publish Plan

What this step does: builds the workflow publish plan for direct small-file publishes and per-split processed sample archives.

Required inputs: dataset_result outputs, DRIVE_PROJECT_ROOT, and WORKTREE_ROOT.

Produced outputs: dataset_publish_plan with workflow-owned direct manifest/report file publish specs and processed sample archive specs.

When this step may be skipped: only when dataset_publish_plan already reflects the current dataset_result, Drive root, runtime root, and split scope.


In [ ]:
dataset_publish_plan = build_dataset_publish_plan(
    dataset_result,
    drive_project_root=DRIVE_PROJECT_ROOT,
    project_root=WORKTREE_ROOT,
)
print("Interim manifest files:", len(dataset_publish_plan.interim_manifest_files))
print("Processed manifest files:", len(dataset_publish_plan.processed_manifest_files))
print("Interim report files:", len(dataset_publish_plan.interim_report_files))
print("Processed report files:", len(dataset_publish_plan.processed_report_files))
print("Processed sample archives:", len(dataset_publish_plan.sample_archives))


### 3.2 Publish Manifest Files

What this step does: executes each workflow-provided direct manifest publish command with byte progress.

Required inputs: dataset_publish_plan interim and processed manifest file specs plus writable mirrored Drive data directories.

Produced outputs: published_manifest_files mapping each workflow spec label to a byte-verified Drive manifest file.

When this step may be skipped: only when the same Drive manifest files already match the current workflow specs byte-for-byte.


In [ ]:
published_manifest_files = {}
for spec_group in (
    dataset_publish_plan.interim_manifest_files,
    dataset_publish_plan.processed_manifest_files,
):
    for label, publish_spec in spec_group.items():
        spec = publish_spec.publish_command
        print(spec.bash)
        !bash -o pipefail -c {spec.arg}

        if globals().get("_exit_code", 1) != 0:
            raise RuntimeError(spec.failure)

        published_manifest_files[label] = publish_spec.target_path
        print(f"{label}: {published_manifest_files[label]}")

print("Published manifest files:", len(published_manifest_files))


### 3.3 Publish Report Files

What this step does: executes each workflow-provided direct split-scoped report publish command with byte progress.

Required inputs: dataset_publish_plan interim and processed report file specs plus writable mirrored Drive data directories.

Produced outputs: published_report_files mapping each workflow spec label to a byte-verified Drive report file.

When this step may be skipped: only when the same Drive report files already match the current workflow specs byte-for-byte.


In [ ]:
published_report_files = {}
for spec_group in (
    dataset_publish_plan.interim_report_files,
    dataset_publish_plan.processed_report_files,
):
    for label, publish_spec in spec_group.items():
        spec = publish_spec.publish_command
        print(spec.bash)
        !bash -o pipefail -c {spec.arg}

        if globals().get("_exit_code", 1) != 0:
            raise RuntimeError(spec.failure)

        published_report_files[label] = publish_spec.target_path
        print(f"{label}: {published_report_files[label]}")

print("Published report files:", len(published_report_files))


### 3.4 Build Processed Sample Archive Member Lists

What this step does: writes per-split tar member list files from workflow sample archive specs derived from the final processed manifests.

Required inputs: dataset_publish_plan.sample_archives.

Produced outputs: sample archive member list files under the runtime dataset artifact area.

When this step may be skipped: only when the member list files already match dataset_publish_plan for this run.


In [ ]:
for split, archive_spec in dataset_publish_plan.sample_archives.items():
    write_dataset_archive_member_list(archive_spec)
    print(f"{split}: {archive_spec.member_list_path} ({archive_spec.expected_member_count} members)")


### 3.5 Create Processed Sample Archives

What this step does: executes each workflow-provided processed sample archive creation command with file-count progress.

Required inputs: dataset_publish_plan.sample_archives and their member list files.

Produced outputs: per-split workflow-verified processed sample archives in Drive.

When this step may be skipped: only when the Drive sample archives already match the current workflow specs.


In [ ]:
for split, archive_spec in dataset_publish_plan.sample_archives.items():
    spec = archive_spec.create_command
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)

    print(f"{split} processed sample archive created:", archive_spec.archive_path)


### 3.6 Verify Processed Sample Archives

What this step does: executes each workflow-provided processed sample archive member listing command with file-count progress and compares observed members exactly to the spec.

Required inputs: dataset_publish_plan.sample_archives and the created processed sample archives.

Produced outputs: verified processed sample archives with exact expected project-relative members.

When this step may be skipped: only when the same sample archives have already passed exact member verification.


In [ ]:
for split, archive_spec in dataset_publish_plan.sample_archives.items():
    spec = archive_spec.verify_command
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)

    print(f"{split} processed sample archive member list observed:", archive_spec.observed_member_list_path)


### 3.7 Inspect Published Dataset Outputs

What this step does: prints workflow-owned Drive-published manifest files, report files, and processed sample archive summaries.

Required inputs: dataset_publish_plan after direct file publish and sample archive verification.

Produced outputs: dataset_publish_summary with published file paths, byte sizes, sample archive paths, archive byte sizes, and expected archive member counts.

When this step may be skipped: only when these published output paths, counts, and sizes have already been inspected for this run.


In [ ]:
dataset_publish_summary = summarize_dataset_publish_plan(dataset_publish_plan)

print("Published manifest files")
print("------------------------")
for spec_group in (
    dataset_publish_plan.interim_manifest_files,
    dataset_publish_plan.processed_manifest_files,
):
    for spec in spec_group.values():
        file_path = dataset_publish_summary.file_paths[spec.label]
        file_size = dataset_publish_summary.file_sizes[spec.label]
        print(f"- {spec.label}: {file_path} ({file_size} bytes)")

print()
print("Published report files")
print("----------------------")
for spec_group in (
    dataset_publish_plan.interim_report_files,
    dataset_publish_plan.processed_report_files,
):
    for spec in spec_group.values():
        file_path = dataset_publish_summary.file_paths[spec.label]
        file_size = dataset_publish_summary.file_sizes[spec.label]
        print(f"- {spec.label}: {file_path} ({file_size} bytes)")

print()
print("Published sample archives")
print("-------------------------")
for label, archive_path in dataset_publish_summary.sample_archive_paths.items():
    member_count = dataset_publish_summary.sample_archive_member_counts[label]
    archive_size = dataset_publish_summary.sample_archive_sizes[label]
    print(f"- {label}: {archive_path} ({member_count} members, {archive_size} bytes)")
